# 01 — Exploration du dataset Sunbird

Goal: understand the distribution of the noise measurements (dB), the geographic coverage,
and check that the dataset is usable as a training base for the surrogate model.

Dataset : https://huggingface.co/datasets/Sunbird/urban-noise-uganda-61k

> Prerequisite: accept the access terms on HuggingFace and create a token at
> https://huggingface.co/settings/tokens

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# HF token loaded from the .env file at the repository root (gitignored)
from dotenv import load_dotenv
import os
load_dotenv('../.env')
HF_TOKEN = os.environ['HF_TOKEN']

ds = load_dataset("Sunbird/urban-noise-uganda-61k", "small", token=HF_TOKEN)
print(ds)

In [ ]:
# Convertit en DataFrame
df = ds['train'].to_pandas()
df = df.dropna(subset=['noise_measurement', 'latitude', 'longitude'])
print(f'{len(df)} rows loaded')
df.head()

## Distribution de noise_measurement

This is the key test: we need good variance (ideally a 20+ dB spread).

In [ ]:
print(df['noise_measurement'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['noise_measurement'], bins=40, ax=axes[0])
axes[0].set_xlabel('Noise level (dB)')
axes[0].set_title('Distribution globale')

sns.boxplot(data=df, x='region', y='noise_measurement', ax=axes[1])
axes[1].set_title('By region')

plt.tight_layout()
plt.savefig('../results/figures/sunbird/sunbird_distribution.png', dpi=150)
plt.show()

## Carte des points de mesure

In [ ]:
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

for _, row in df.sample(min(500, len(df))).iterrows():
    color = 'green' if row['noise_measurement'] < 60 else ('orange' if row['noise_measurement'] < 75 else 'red')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color=color,
        fill=True,
        popup=f"{row['noise_measurement']:.1f} dB — {row['class']}"
    ).add_to(m)

m.save('../results/figures/sunbird/sunbird_map.html')
print('Map saved to results/figures/sunbird/sunbird_map.html')
m

## Conclusion

- **Enough variance?** If stdev > 8 dB, yes, the model has something to learn
- **Balanced distribution?** Check that 90% of the points are not in a single band
- **Geographic coverage?** Are residential, commercial and main-road areas well represented?